# ![](https://ga-dash.s3.amazonaws.com/production/assets/logo-9f88ae6c9c3871690e33280fcf557f33.png) 4.03 - Generalized Linear Models
#### Creator: Matt Brems (DC) h/t Timothy Book.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

import matplotlib.pyplot as plt
%matplotlib inline

## Generalized Linear Models (GLMs)
**Generalized linear models** describes a class of models that take the linear model (think linear regression) and generalizes the linear model beyond the assumptions of the linear regression model we discussed last week.

## The `statsmodels` API
In what follows, we will explore the statsmodels API, which is a one-stop-shop for most things GLM in Python.  Documentation and examples can be found [here](http://www.statsmodels.org/stable/generated/statsmodels.genmod.generalized_linear_model.GLM.html#statsmodels.genmod.generalized_linear_model.GLM).

## Logistic Regression

**When do we use it?** When we want to model something on the $\{0,1\}$ range... like whether or not somebody will develop a disease.

### The Data
The data used from this example come from one of UCLA's amazing IDRE modules (for R).  The module can be found [here](https://stats.idre.ucla.edu/r/dae/logit-regression/).

**Data Description:** _A researcher is interested in how variables, such as GRE (Graduate Record Exam scores), GPA (grade point average) and prestige of the undergraduate institution, effect admission into graduate school. The response variable, admit/don’t admit, is a binary variable._

In [ ]:
grad = pd.read_csv("https://stats.idre.ucla.edu/stat/data/binary.csv")

# Let's do some EDA!
print(grad.shape)
grad.head()

In [ ]:
grad.describe()

In [ ]:
grad.admit.value_counts()

In [ ]:
grad.boxplot('gre', by = 'admit')
grad.boxplot(['gpa', 'rank'], by = 'admit')

In [ ]:
# Now, let's build our GLM
indep_vars = ['gre', 'gpa', 'rank']
x_mat = sm.add_constant(grad[indep_vars])
y_vec = grad.admit
glm_logit = sm.GLM(y_vec, 
                   x_mat,
                   sm.families.Binomial(sm.families.links.logit)).fit()

In [ ]:
glm_logit.summary()

**Check:** How would I interpret the coefficient for `rank`?

In [ ]:
import math
math.exp(-0.5600)

**Check:** If I scored 100 points better on my GRE, how would that affect my odds of acceptance?

In [ ]:
math.exp(100*0.0023)

**Note:**
1. `IRLS` and `No. Iterations.`
    - $\mathbf{\hat{\beta}}_1 = (X^TW_1X)^{-1}(X^TW_1y)$
    - $\mathbf{\hat{\beta}}_2 = (X^TW_2X)^{-1}(X^TW_2y)$
    - $\mathbf{\hat{\beta}}_3 = (X^TW_3X)^{-1}(X^TW_3y)$
    - $\vdots$
    - Once the difference between $\mathbf{\hat{\beta}}_i$ and $\mathbf{\hat{\beta}}_{i+1}$ is small enough, the algorithm has "converged" and we have our coefficient estimates for $\mathbf{\beta}$.
    - The maximum number of iterations for GLMs is 100. If `No. Iterations` is 100, that means the algorithm probably didn't converge and that the $\mathbf{\hat{\beta}}$ are still changing. Therefore, **your output is unreliable - DO NOT USE IT**.
2. Does this model perform well?

In [ ]:
# Next, let's see if we can safely reduce our model
reduced_vars = ['gre', 'gpa']
x_reduced = sm.add_constant(grad[reduced_vars])
glm_reduced = sm.GLM(y_vec,
                    x_reduced,
                    sm.families.Binomial(sm.families.links.logit)).fit()
glm_reduced.summary()

### Analysis of Deviance

We've spoken briefly about deviance before as a generalization of the sums of squares of error for generalized linear models.

Suppose we have two models:

$$
\begin{eqnarray*}
Y_{full} &=& \beta_0 + \beta_1X_1 + \cdots + \beta_kX_k + \cdots + \beta_pX_p + \varepsilon \\
Y_{reduced} &=& \beta_0 + \beta_1X_1 + \cdots + \beta_kX_k + \varepsilon
\end{eqnarray*}
$$

We say that $Y_{reduced}$ is nested in $Y_{full}$, because the reduced model could "fit inside" the full. (You can learn more about nested linear regression models [here](http://people.reed.edu/~jones/Courses/P24.pdf), although the ideas approximately hold for generalized linear models as well.)

When we have one model nested inside the other, there is a statistical test to see if adding new variables are worth it. (Think about it like looking at the difference in mean squared error or $R^2$ by adding a variable, but getting a $p$-value quantifying whether or not it's worth it!)

We calculate the **deviance** of the reduced model and subtract the **deviance** of the full model from it. This difference in deviance follows a [chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) with $p-k$ degrees of freedom. (Note that $p-k$ indicates how many variables we took out of our full model to get to the reduced model!)

**This comparison only works with nested models! Do not use if your models are not nested!**

In [ ]:
# Test model differences
from scipy.stats import chi2

D = glm_reduced.deviance - glm_logit.deviance
print('Difference in Deviance: ', D)
pval = 1 - chi2.cdf(D, df = 1)
print('p-value of test of difference: ', pval) # What can we conclude here?

## Poisson Regression

**When do we use it?** When we want to model something on the $\{0,1,2,\ldots\}$ range... like number of objects sold or nunber of awards earned!

#### Data
We'll again rely on UCLA's IDRE module.  This one can be found [here](https://stats.idre.ucla.edu/r/dae/poisson-regression/).

#### Data Description
_The number of awards earned by students at one high school. Predictors of the number of awards earned include the type of program in which the student was enrolled (e.g., vocational, general or academic) and the score on their final exam in math._

In [ ]:
award = pd.read_csv("https://stats.idre.ucla.edu/stat/data/poisson_sim.csv")

# Let's do some EDA:
award.head()

In [ ]:
award.plot('math', 'num_awards', kind = 'scatter')
award.boxplot('num_awards', by = 'prog')

In [ ]:
# Notice that prog is actually a categorical variable - I am aware of this.
# I'm going to suspsend that knowledge for the sake of example.
poi_vars = ['prog', 'math']
x_poi = sm.add_constant(award[poi_vars])
y_poi = award.num_awards

glm_poi = sm.GLM(y_poi,
                 x_poi,
                 family = sm.families.Poisson()).fit()
glm_poi.summary()

**Interpretation**: For a one-unit increase in `math`, I expect to win $e^{0.0861} \approx 1.09$ times as many awards.

In [ ]:
math.exp(0.0861)

In [ ]:
# Interesting - it looks like prog is pretty insignificant.
# Let's try removing it and seeing what happens.
x_poi_red = sm.add_constant(award.math)

glm_poi_red = sm.GLM(y_poi,
                     x_poi_red,
                     family = sm.families.Poisson()).fit()
glm_poi_red.summary()

In [ ]:
# Test model difference:
D_poi = glm_poi_red.deviance - glm_poi.deviance
print('Difference in Deviance: ', D_poi)
pval_poi = 1 - chi2.cdf(D_poi, 1)
print('p-value of test of difference: ', pval_poi) # What does this mean?

## Gamma Regression

**When do we use it?** When we want to model something on the $[0,\infty)$ range... like time until some event occurs!

### The Data
The data used from this example come from a 1945 study about and is inspired by [Peter Craigmile's use](http://www.craigmile.com/peter/teaching/7430/notes/7_gamma_influence.pdf) of this example.

**Data Description:** _“Hurn, et al. (1945) published data on the clotting time of blood, giving clotting time in seconds for normal plasma diluted to nine different percentage concentrations with prothrombin-free plasma; clotting was induced by two lots of thromboplastin.” [McCullagh and Nelder](http://www.utstat.toronto.edu/~brunner/oldclass/2201s11/readings/glmbook.pdf)_

In [ ]:
clot = pd.read_csv("./clotting.csv", index_col = "index")

In [ ]:
clot.head()

In [ ]:
clot.boxplot('clot_time', by = 'lot')

In [ ]:
clot.plot('plasma_pct', 'clot_time', kind = 'scatter')

In [ ]:
clot['plasma_pct_log10'] = np.log10(clot['plasma_pct'])

In [ ]:
clot.plot('plasma_pct_log10', 'clot_time', kind = 'scatter')

In [ ]:
clot['interaction'] = clot['plasma_pct_log10'] * clot['lot']

In [ ]:
y_gam = clot['clot_time']
x_gam = sm.add_constant(clot[['plasma_pct_log10','lot','interaction']])

glm_gam = sm.GLM(y_gam,
                 x_gam,
                 family = sm.families.Gamma()).fit()
glm_gam.summary()

In [ ]:
y_gam = clot['clot_time']
x_gam = sm.add_constant(clot[['plasma_pct_log10','lot','interaction']])

glm_gam_log = sm.GLM(y_gam,
                     x_gam,
                     family = sm.families.Gamma(link = sm.families.links.log)).fit()
glm_gam_log.summary()

**Interpretation**: For a one-unit increase in `plasma_pct_log10`, I expect the blood will take $e^{-1.4654} \approx 23\%$ as much time to clot.

In [ ]:
math.exp(-1.4654)